## 1. Import

In [40]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from typing import Set, List, Dict

# Set random seed for reproducibility
RANDOM_SEED = 41
np.random.seed(RANDOM_SEED)

## 2. Load Dataset

In [41]:
# Load the dataset
data_path = Path('../../data/all_recipes_final.csv')
df = pd.read_csv(data_path)

print(f"Dataset loaded: {len(df)} recipes")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Dataset loaded: 10263 recipes

Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']

Missing values:
title                        0
type_of_food                 0
link                         0
description                  7
ingredients                  0
ingredients_normalized       0
step                         0
note                         0
num_of_ingredients           0
cook_time                  295
num_of_people              294
calories                  9826
source                       0
dtype: int64


In [42]:
# Handle missing data
# Drop rows with missing ingredients or type_of_food
df_clean = df.dropna(subset=['ingredients', 'type_of_food']).copy()

# Fill missing titles with placeholder
if 'title' in df_clean.columns:
    df_clean['title'] = df_clean['title'].fillna('Untitled Recipe')

# Reset index and create a unique ID
df_clean = df_clean.reset_index(drop=True)
df_clean['recipe_id'] = df_clean.index

print(f"After cleaning: {len(df_clean)} recipes")
print(f"Dropped {len(df) - len(df_clean)} recipes with missing critical data")

After cleaning: 10263 recipes
Dropped 0 recipes with missing critical data


In [43]:
# Convert ingredients_normalized from string to actual set objects
import ast

def parse_ingredients_set(ingredient_str):
    """Parse string representation of set to actual set object"""
    try:
        if isinstance(ingredient_str, str):
            return ast.literal_eval(ingredient_str)
        elif isinstance(ingredient_str, set):
            return ingredient_str
        else:
            return set()
    except:
        return set()

df_clean['ingredients_normalized'] = df_clean['ingredients_normalized'].apply(parse_ingredients_set)

print(f"Converted ingredients_normalized to set objects")
print(f"Sample value: {df_clean['ingredients_normalized'].iloc[0]}")
print(f"Type: {type(df_clean['ingredients_normalized'].iloc[0])}")

Converted ingredients_normalized to set objects
Sample value: {'tro bếp hoặc nước vo gọa', 'cà rốt trang trí tùy chọn', 'lọ sạch', 'muối hạt', 'hành củ tươi', 'đường'}
Type: <class 'set'>


## 4. Select Random Queries

In [44]:
# Configuration
NUM_QUERIES = 200
TOP_K = 10

# Select random queries
query_indices = np.random.choice(df_clean.index, size=NUM_QUERIES, replace=False)

queries_df = df_clean.loc[query_indices].copy()
print(f"Selected {len(queries_df)} queries")
print(f"\nType of food distribution in queries:")
print(queries_df['type_of_food'].value_counts())

Selected 200 queries

Type of food distribution in queries:
type_of_food
Món bánh                  40
Món chiên                 23
Món kho                   16
Ăn vặt                    13
Món xào                   13
Thức uống                 10
Món ngon hàng ngày         9
Món canh                   8
Món chay                   7
Món gỏi - salad            7
Món hấp                    6
Món nướng                  6
Món tráng miệng            4
Món chính                  4
Món cuốn - trộn            4
Món nước                   4
Món cháo                   3
Món kem                    3
Món từ bò                  3
Món khô - mắm              3
Bữa sáng đơn giản          2
Trà sữa                    2
Ngày lễ Tết                2
Món chè                    2
Món Tết                    1
Món ngon ngày lạnh         1
Món từ gà                  1
Món lẩu                    1
Món ngon cho cuối tuần     1
Quà - Món ăn vặt           1
Name: count, dtype: int64


In [45]:
queries_df.head()

,title,type_of_food,link,description,ingredients,ingredients_normalized,step,note,num_of_ingredients,cook_time,num_of_people,calories,source,recipe_id
6302,Cá rô kho gừng thơm ngon đậm đà hương vị cho b...,Món kho,https://www.dienmayxanh.com/vao-bep/cach-lam-c...,"Vào những ngày mưa gió, được quây quần cùng gi...","['600 gr Cá rô', '2 củ Gừng', '1 muỗng canh Dầ...","{cá rô, nước mắm, ớt hiểm, gừng, nước màu điều...",['Bước 1: Sơ chế cá rô: Cá rô mua về về tiến h...,['Xem chi tiết: Cách chọn mua cá rô ngon đúng ...,7,60 phút,3 người,NaN,dienmayxanh,6302
3779,Bánh táo yến mạch đơn giản thơm mềm chiêu đãi ...,Món bánh,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,Các món bánh nướng thơm ngon ngọt ngào thường ...,"['1 quả Táo đỏ', '2 quả Trứng gà', '70 gr Yến ...","{nho khô/ hạnh nhân, bơ đậu phộng, trứng gà, d...","['Bước 1: Sơ chế táo: Táo sau khi mua về, bạn ...",['Xem chi tiết: Cách chọn mua táo tươi ngon kh...,11,55 phút,4 người,NaN,dienmayxanh,3779
768,Bỏ túi cách làm món khoai chiên mayo lắc mè cự...,Món tráng miệng,https://vncooking.com/cong-thuc/bo-tui-cach-la...,Mách bạn cách làm món khoai chiên mayo lắc mè ...,"['khoai lang 100 gam', 'Khoai mì 100 gam', 'Kh...","{khoai lang am, lá quế am, khoai tây am, khoai...","['Bước 1: Chuẩn bị nguyên liệu Đầu tiên, bạn t...",[],4,30phút,2,NaN,vncooking,768
3399,Bánh sắn chiên cốt dừa thơm ngon giòn rụm,Món bánh,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,"Bánh sắn chiên cốt dừa là món ăn thơm ngon, hấ...","['400 gr Củ sắn', '100 gr Dừa nạo sợi', '50 gr...","{nước cốt dừa, dừa nạo sợi, sữa đặc, củ sắn, m...",['Bước 1: Luộc sắn: Củ sắn mua về các bạn dùng...,"['', 'Dùng chày giã để tăng độ dẻo của củ sắn ...",6,30 phút,4 người,NaN,dienmayxanh,3399
4561,"Kem dừa không Whipping Cream, không sữa đặc, k...",Món kem,https://www.dienmayxanh.com/vao-bep/cach-lam-k...,Chẳng cần đến máy làm kem hay nguyên liệu phức...,"['800 ml Nước cốt dừa', '12 gr Bột bắp', '100 ...","{nước cốt dừa, muối, đường, bột bắp}",['Bước 1: Trộn hỗn hợp nước cốt dừa: Cho vào n...,['Việc xay kem lại bằng máy xay sinh tố sẽ giú...,4,15 phút,2 người,NaN,dienmayxanh,4561


## 5. Calculate Jaccard Similarity and Assign Relevance Grades

In [46]:
def jaccard_similarity(set1: Set[str], set2: Set[str]) -> float:
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1 & set2) # số nguyên liệu trùng (giao)
    union = len(set1 | set2)        # tổng số nguyên liệu (hợp)
    return intersection / union if union > 0 else 0.0

def assign_relevance_grade(jaccard_score: float) -> int:
    """
    Assign graded relevance based on Jaccard similarity.
    - < 0.10: 0 (not relevant)
    - 0.10 - 0.19: 1 (somewhat relevant)
    - 0.20 - 0.29: 2 (relevant)
    - >= 0.30: 3 (highly relevant)
    """
    if jaccard_score < 0.10:
        return 0
    elif jaccard_score < 0.20:
        return 1
    elif jaccard_score < 0.30:
        return 2
    else:
        return 3

In [47]:
test_set1 = {'a', 'b', 'c', 'd'}
test_set2 = {'b', 'c', 'e', 'f'}
test_jaccard = jaccard_similarity(test_set1, test_set2)
print(f"Test Jaccard: {test_jaccard:.3f}")
print(f"Test Relevance Grade: {assign_relevance_grade(test_jaccard)}")

Test Jaccard: 0.333
Test Relevance Grade: 3


## 6. Build Evaluation Data for All Queries

In [48]:
def find_top_k_similar(query_row, candidates_df, k=10):
    """
    Find top K most similar items to the query.
    Only considers candidates with the same type_of_food.
    Excludes the query itself from candidates.
    """
    query_id = query_row['recipe_id']
    query_type = query_row['type_of_food']
    query_ingredients = query_row['ingredients_normalized']
    
    # Filter candidates: same type, exclude query itself
    candidates = candidates_df[
        (candidates_df['type_of_food'] == query_type) &    # phải cùng type
        (candidates_df['recipe_id'] != query_id)            # tránh so sánh với chính nó
    ].copy()
    
    if len(candidates) == 0:
        return []
    
    # Calculate Jaccard similarity for all candidates
    candidates['jaccard'] = candidates['ingredients_normalized'].apply(
        lambda x: jaccard_similarity(query_ingredients, x)
    )
    
    # Sort by Jaccard and take top K
    top_k = candidates.nlargest(k, 'jaccard')
    
    # Prepare results
    results = []
    for _, row in top_k.iterrows():
        jaccard_score = row['jaccard']
        results.append({
            'doc_id': int(row['recipe_id']),
            'doc_title': row['title'] if 'title' in row else 'Untitled',
            'jaccard': float(jaccard_score),
            'rel': assign_relevance_grade(jaccard_score)
        })
    
    return results

In [49]:
# Process all queries (with progress indicator)
eval_data = []

for idx, (_, query_row) in enumerate(queries_df.iterrows()):
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(queries_df)} queries...")
    
    top_k_results = find_top_k_similar(query_row, df_clean, k=TOP_K)
    
    eval_entry = {
        'query_id': int(query_row['recipe_id']),
        'query_title': query_row['title'] if 'title' in query_row else 'Untitled',
        'query_type': query_row['type_of_food'],
        'query_ingredients': list(query_row['ingredients_normalized']),
        'top10': top_k_results
    }
    
    eval_data.append(eval_entry)

  Processed 50/200 queries...
  Processed 100/200 queries...
  Processed 150/200 queries...
  Processed 200/200 queries...


## 7. Inspect Sample Results

In [50]:
# Print first 2 queries as examples
for i in range(min(2, len(eval_data))):
    query = eval_data[i]
    print(f"\n{'='*80}")
    print(f"QUERY {i+1}")
    print(f"{'='*80}")
    print(f"Query ID: {query['query_id']}")
    print(f"Query Title: {query['query_title']}")
    print(f"Query Type: {query['query_type']}")
    print(f"Query Ingredients ({len(query['query_ingredients'])}): {query['query_ingredients']}")
    print(f"\nTop 10 Similar Items:")
    print(f"{'Rank':<6} {'Doc ID':<10} {'Jaccard':<10} {'Rel':<6} {'Title'}")
    print("-" * 80)
    
    for rank, item in enumerate(query['top10'], 1):
        title_short = item['doc_title']
        print(f"{rank:<6} {item['doc_id']:<10} {item['jaccard']:<10.3f} {item['rel']:<6} {title_short}")


QUERY 1
Query ID: 6302
Query Title: Cá rô kho gừng thơm ngon đậm đà hương vị cho bữa cơm
Query Type: Món kho
Query Ingredients (7): ['cá rô', 'nước mắm', 'ớt hiểm', 'gừng', 'nước màu điều', 'gia vị thông dụng muối/ đường/ tiêu/ hạt nêm/ bột ngọt', 'dầu ăn']

Top 10 Similar Items:
Rank   Doc ID     Jaccard    Rel    Title
--------------------------------------------------------------------------------
1      6305       0.333      3      Cá thu kho gừng cay cay ngon miệng đậm đà dễ làm
2      6264       0.308      3      Cá rô kho tộ thơm ngon đậm đà hấp dẫn cực đưa cơm
3      6530       0.300      3      Món cá trê kho tiêu thơm ngon khó cưỡng cực đưa cơm
4      6086       0.273      2      Thịt kho nghệ thơm ngon đậm đà dễ làm cho bữa cơm
5      6319       0.273      2      Cá ngát kho gừng thơm ngon, đậm đà cho bữa cơm thêm tròn vị
6      6290       0.250      2      Món cá diếc kho tiêu thơm ngon đậm đà hấp dẫn tại nhà
7      6446       0.250      2      Thịt kho củ sắn (củ đậu) đậm

## 8. Export Evaluation Data to JSONL

In [51]:
# Export to JSONL file
output_file = Path('eval_ground_truth.jsonl')

with open(output_file, 'w', encoding='utf-8') as f:
    for entry in eval_data:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Evaluation data exported to: {output_file}")
print(f"Total entries: {len(eval_data)}")

Evaluation data exported to: eval_ground_truth.jsonl
Total entries: 200


## 9. Summary Statistics

In [52]:
# Calculate statistics
all_jaccard_scores = []
all_relevance_grades = []

for query in eval_data:
    for item in query['top10']:
        all_jaccard_scores.append(item['jaccard'])
        all_relevance_grades.append(item['rel'])

print(f"\nTotal queries: {len(eval_data)}")
print(f"Total relevance judgments: {len(all_jaccard_scores)}")
print(f"\nJaccard Similarity Statistics:")
print(f"  Mean: {np.mean(all_jaccard_scores):.3f}")
print(f"  Median: {np.median(all_jaccard_scores):.3f}")
print(f"  Std: {np.std(all_jaccard_scores):.3f}")
print(f"  Min: {np.min(all_jaccard_scores):.3f}")
print(f"  Max: {np.max(all_jaccard_scores):.3f}")

print(f"\nRelevance Grade Distribution:")
grade_counts = pd.Series(all_relevance_grades).value_counts().sort_index()
for grade, count in grade_counts.items():
    percentage = count / len(all_relevance_grades) * 100
    print(f"  Grade {grade}: {count:4d} ({percentage:5.1f}%)")


Total queries: 200
Total relevance judgments: 1996

Jaccard Similarity Statistics:
  Mean: 0.201
  Median: 0.200
  Std: 0.112
  Min: 0.000
  Max: 1.000

Relevance Grade Distribution:
  Grade 0:  296 ( 14.8%)
  Grade 1:  689 ( 34.5%)
  Grade 2:  659 ( 33.0%)
  Grade 3:  352 ( 17.6%)
